In [2]:
# cargamos los datos de la API

import requests
import pandas as pd

url = "https://api.bcra.gob.ar/estadisticas/v4.0/monetarias/27"
r=requests.get(url, params={"desde":"2017-01-01","hasta": "2026-09-30"})
r.status_code # 200 si sale bien, 404 o 500 si no

200

In [4]:
datos=r.json()["results"][0]["detalle"] # json convierte en estructuras que Python lee
# detalles es la listas de pares fecha-valor
df=pd.DataFrame(datos) # convierte lista en tabla de pares, fecha y valor quedan como columnas
df["fecha"]=pd.to_datetime(df["fecha"])
df=df.sort_values("fecha").reset_index(drop=True) # fechas de mas antigua a mas reciente y desechamos las viejas enumeraciones
df.head()


,fecha,valor
0,2017-01-31,1.3
1,2017-02-28,2.5
2,2017-03-31,2.4
3,2017-04-30,2.6
4,2017-05-31,1.3


In [ ]:
print(df.shape)
print(df.dtypes)
print(df["valor"].isna().sum())
print(df["fecha"].is_monotonic_increasing) # la fecha va subiendo? esperamos True

(117, 2)
fecha    datetime64[ns]
valor           float64
dtype: object
0
True


In [8]:
from pathlib import Path

Path("../data/raw").mkdir(parents=True, exist_ok=True)
df.to_csv("../data/raw/bcra_inflacion_mensual.csv", index=False)

In [9]:
pd.read_csv("../data/raw/bcra_inflacion_mensual.csv").tail()

,fecha,valor
112,2026-05-31,2.1
113,2026-06-30,1.9
114,2026-07-31,2.1
115,2026-08-31,1.7
116,2026-09-30,1.9


In [15]:
def descargar_serie(id_variable, desde, hasta):
    url = f"https://api.bcra.gob.ar/estadisticas/v4.0/monetarias/{id_variable}"
    limite = 1000
    offset = 0 # porque sino la serie del fx se corta en 1000, llamamos por tandas
    filas = []
    while True:
        r = requests.get(url, params={"desde": desde, "hasta": hasta,
                                      "limit": limite, "offset": offset})
        r.raise_for_status() # a ver si salió bien
        detalle = r.json()["results"][0]["detalle"]
        filas.extend(detalle)
        if len(detalle) < limite:
            break
        offset += limite
    df = pd.DataFrame(filas)
    df["fecha"] = pd.to_datetime(df["fecha"])
    return df.sort_values("fecha").reset_index(drop=True)


In [16]:
df2 = descargar_serie(27, "2017-01-01", "2026-09-30")
df2.shape

(117, 2)

In [17]:
# el TC mayorista es con el que comercian bancos y empresas (minorista=mayorista+margen), y en Argentina
# es importante para analizar la inflación por el pass-through (efecto del cambio en TC en los precios internos)

# con la serie diaria tenemos los datos de todo el mes, y como pronosticamos despues de que el mes haya
# terminado, conocemos el TC de todos esos dias aunque el INDEC no haya publicado el IPC. Es 
# la ventaja informativa del nowcast.

fx=descargar_serie(5, "2017-01-01", "2026-11-03")
print(fx.shape)
print(fx["fecha"].min(), fx["fecha"].max())

(2391, 2)
2017-01-02 00:00:00 2026-11-03 00:00:00


In [19]:
# BADLAR: es la tasa que pagan los bancos privados por plazos fijos grandes. Es una medida de las condiciones
# monetarias en pesos y de lo que el mercado espera. Si los ahorristas piden tasas más altas, suele ser 
# porque esperan más inflación o más depreciación. Además, comparada con la inflación, da la tasa real: si la 
# tasa queda por debajo de lo que sube el costo de vida, hay presión hacia el dólar.

series={
    "tc_mayorista":5,
    "badlar_nominal":7,
    "reservas":1,
    "inflacion_mensual":27
}

for nombre, id_variable in series.items():
    d=descargar_serie(id_variable, "2017-01-01", "2026-11-03")
    d.to_csv(f"../data/raw/bcra_{nombre}.csv", index=False)
    print(nombre, d.shape, d["fecha"].min().date(), d["fecha"].max().date(),
          "duplicados:", d["fecha"].duplicated().sum())

tc_mayorista (2391, 2) 2017-01-02 2026-11-03 duplicados: 0
badlar_nominal (2391, 2) 2017-01-02 2026-11-03 duplicados: 0
reservas (2390, 2) 2017-01-02 2026-11-02 duplicados: 0
inflacion_mensual (117, 2) 2017-01-31 2026-09-30 duplicados: 0


In [21]:
# traemos el indice del ipc nacional de datos.gob.ar (otra api, otros parametros)
url_ipc = "https://apis.datos.gob.ar/series/api/series/"
r = requests.get(url_ipc, params={"ids": "148.3_INIVELNAL_DICI_M_26",  # ipc nacional nivel general
                                  "start_date": "2016-12-01",  # desde dic-2016 para poder calcular enero
                                  "limit": 1000,
                                  "format": "json"})
r.raise_for_status()  # a ver si salio bien

# los datos vienen como lista de pares [fecha, valor], les ponemos nombre a las columnas
ipc = pd.DataFrame(r.json()["data"], columns=["fecha", "indice"])
ipc["fecha"] = pd.to_datetime(ipc["fecha"])

# variacion mensual en %: cuanto subio el indice respecto del mes anterior
ipc["var_mensual"] = ipc["indice"].pct_change() * 100

# ultimas 5 filas para chequear
ipc.tail()

,fecha,indice,var_mensual
112,2026-04-01,11363.0904,2.582180
113,2026-05-01,11607.3937,2.149972
114,2026-06-01,11826.4103,1.886871
115,2026-07-01,12076.3937,2.113772
116,2026-08-01,12276.7660,1.659206


In [22]:
# leemos la inflacion que guardamos del BCRA
infl = pd.read_csv("../data/raw/bcra_inflacion_mensual.csv", parse_dates=["fecha"])

# las fechas no coinciden (una es fin de mes, la otra inicio), asi que creamos "mes" para poder cruzarlas
ipc["mes"] = ipc["fecha"].dt.to_period("M")
infl["mes"] = infl["fecha"].dt.to_period("M")

# juntamos las dos tablas por mes, como un buscarv
comp = ipc.merge(infl[["mes", "valor"]], on="mes")

# diferencia entre lo que calculamos y lo publicado
comp["dif"] = (comp["var_mensual"].round(1) - comp["valor"]).abs()

# a ver si coinciden
print(comp.shape)  # tienen que ser ~116 filas
print(comp["dif"].max())  # la peor diferencia
print((comp["dif"] > 0.05).sum())  # cuantos meses difieren
comp[comp["dif"] > 0.05]  # cuales son (vacio = todo bien)

(116, 6)
0.3999999999999999
4


,fecha,indice,var_mensual,mes,valor,dif
0,2017-01-01,101.5859,1.585900,2017-01,1.3,0.3
1,2017-02-01,103.6859,2.067216,2017-02,2.5,0.4
3,2017-04-01,108.9667,2.655830,2017-04,2.6,0.1
4,2017-05-01,110.5301,1.434750,2017-05,1.3,0.1


In [23]:
# leemos el tipo de cambio mayorista diario
tc = pd.read_csv("../data/raw/bcra_tc_mayorista.csv", parse_dates=["fecha"])
tc["mes"] = tc["fecha"].dt.to_period("M")

# agrupamos por mes y resumimos cada mes con el promedio y el ultimo dato
tc_mensual = tc.groupby("mes")["valor"].agg(tc_promedio="mean", tc_ultimo="last")

# variacion % del promedio respecto al mes anterior
tc_mensual["tc_var"] = tc_mensual["tc_promedio"].pct_change() * 100
tc_mensual.tail()

,tc_promedio,tc_ultimo,tc_var
mes,,,
2026-07,1487.003810,1488.4531,2.532698
2026-08,1499.962935,1509.4686,0.871492
2026-09,1513.886973,1517.5209,0.928292
2026-10,1525.019143,1537.3487,0.735337
2026-11,1547.185050,1549.7194,1.453484


In [24]:
# convierte una serie diaria guardada en csv a una tabla mensual
def a_mensual(nombre, prefijo, como="pct"):
    d = pd.read_csv(f"../data/raw/bcra_{nombre}.csv", parse_dates=["fecha"])
    d["mes"] = d["fecha"].dt.to_period("M")

    # promedio y ultimo dato de cada mes, con el prefijo para saber de que serie son
    m = d.groupby("mes")["valor"].agg(promedio="mean", ultimo="last").add_prefix(f"{prefijo}_")

    # como medimos el cambio: en % (para niveles) o en puntos (para tasas)
    if como == "pct":
        m[f"{prefijo}_var"] = m[f"{prefijo}_promedio"].pct_change() * 100
    else:
        m[f"{prefijo}_var"] = m[f"{prefijo}_promedio"].diff()
    return m

tc_m = a_mensual("tc_mayorista", "tc")
badlar_m = a_mensual("badlar_nominal", "badlar", como="diff")
reservas_m = a_mensual("reservas", "res")

# juntamos las tres tablas por mes
base = tc_m.join(badlar_m).join(reservas_m)
base.tail()

,tc_promedio,tc_ultimo,tc_var,badlar_promedio,badlar_ultimo,badlar_var,res_promedio,res_ultimo,res_var
mes,,,,,,,,,
2026-07,1487.003810,1488.4531,2.532698,21.318452,21.6250,0.627976,48628.714286,47599.0,2.414948
2026-08,1499.962935,1509.4686,0.871492,22.721875,23.0000,1.403423,49858.750000,48259.0,2.529443
2026-09,1513.886973,1517.5209,0.928292,22.468750,22.8125,-0.253125,49633.318182,46092.0,-0.452141
2026-10,1525.019143,1537.3487,0.735337,22.625000,22.7500,0.156250,49725.714286,48565.0,0.186157
2026-11,1547.185050,1549.7194,1.453484,22.531250,22.7500,-0.093750,50439.000000,50439.0,1.434440


In [25]:
# la inflacion del bcra es lo que queremos predecir
infl = pd.read_csv("../data/raw/bcra_inflacion_mensual.csv", parse_dates=["fecha"])
infl["mes"] = infl["fecha"].dt.to_period("M")
objetivo = infl.set_index("mes")["valor"].rename("inflacion")

# juntamos todo en una sola tabla por mes
tabla = base.join(objetivo)

# sacamos ene-may 2017 (las dos fuentes no coincidian ahi)
tabla = tabla.loc["2017-06":]

# guardamos la tabla lista para modelar
from pathlib import Path
Path("../data/processed").mkdir(parents=True, exist_ok=True)
tabla.to_csv("../data/processed/tabla_mensual.csv")

print(tabla.shape)
tabla[["tc_var", "badlar_var", "res_var", "inflacion"]].tail()

(114, 10)


,tc_var,badlar_var,res_var,inflacion
mes,,,,
2026-07,2.532698,0.627976,2.414948,2.1
2026-08,0.871492,1.403423,2.529443,1.7
2026-09,0.928292,-0.253125,-0.452141,1.9
2026-10,0.735337,0.156250,0.186157,NaN
2026-11,1.453484,-0.093750,1.434440,NaN
